# CREACIÓN DE UN MODELO USANDO TRANSFORMERS DE HUGGING FACE

# PASO 1 - INSTALAR TRANSFORMERS Y DATASETS DE HUGGING FACE

In [1]:
!pip install transformers==4.48.3 datasets==3.5.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.2
    Uninstalling transformers-4.53.2:
      Successfully uninstalled transformers-4.53.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tor

# PASO 2 - IMPORTAR LIBRERIAS

In [2]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# PASO 3 - CARGAMOS EL DATASET SMS_SPAM_COLLECTION

In [3]:
dataset = load_dataset("codesignal/sms-spam-collection")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sms-spam-collection.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5572 [00:00<?, ? examples/s]

# PASO 4 - PREPROCESAR LOS DATOS Y DIVIDIMOS DATASET EN TRAIN Y TEST

In [4]:
df = dataset['train'].to_pandas()
df

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [5]:
# Dividir en entrenamiento y validación (80% entrenamiento, 20% validación)
train_df, valid_df = train_test_split(df, test_size=0.2, random_state=42)

In [6]:
from datasets import Dataset
# Convertir de nuevo a datasets de Hugging Face
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

In [7]:
# Asegurarse de que las etiquetas sean enteros (0 para 'ham', 1 para 'spam')
def preprocess_labels(examples):
    examples['label'] = [1 if label == 'spam' else 0 for label in examples['label']]
    return examples

# Aplicar la transformación de las etiquetas
train_dataset = train_dataset.map(preprocess_labels, batched=True)
valid_dataset = valid_dataset.map(preprocess_labels, batched=True)

Map:   0%|          | 0/4457 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

In [8]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
  return tokenizer(examples['message'],padding='max_length',truncation=True,max_length=128)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [9]:
# Aplicar la tokenización a los datos de entrenamiento y validación
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4457 [00:00<?, ? examples/s]

Map:   0%|          | 0/1115 [00:00<?, ? examples/s]

# PASO 5 - CREAR EL MODELO EN BASE A BERT PARA LA CLASIFICIÓN

In [10]:
# Cargar el modelo preentrenado para clasificación de secuencias
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
# Mover el modelo a GPU si está disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

# PASO 6 - CONFIGURAR EL MODELO

In [12]:
training_args = TrainingArguments(
    output_dir='./results',          # Directorio donde se guardarán los resultados
    evaluation_strategy="epoch",     # Evaluación al final de cada época
    learning_rate=2e-5,              # Tasa de aprendizaje
    per_device_train_batch_size=8,   # Tamaño del batch de entrenamiento
    per_device_eval_batch_size=8,    # Tamaño del batch de evaluación
    num_train_epochs=3,              # Número de épocas
    weight_decay=0.01,               # Decaimiento de peso
    logging_dir='./logs',            # Directorio para guardar los logs
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


# PASO 7 - ENTRENAMOS EL MODELO

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
)

# Entrenar el modelo
trainer.train()

/tmp/ipython-input-13-2496162404.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cmaytadatag4 (cmaytadatag4-codigo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.087200,0.030286
2,0.026200,0.047446
3,0.004300,0.065210


TrainOutput(global_step=1674, training_loss=0.0353199939645177, metrics={'train_runtime': 482.4952, 'train_samples_per_second': 27.712, 'train_steps_per_second': 3.469, 'total_flos': 879514480304640.0, 'train_loss': 0.0353199939645177, 'epoch': 3.0})

# PASO 8 - EVALUAMOS EL MODELO

In [14]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.06520991027355194, 'eval_runtime': 7.4229, 'eval_samples_per_second': 150.21, 'eval_steps_per_second': 18.86, 'epoch': 3.0}


# PASO 9 - PROBAMOS EL CLASIFICADOR

In [16]:
# Función para predecir si un email es spam o no
def predict_email(email_text):
    inputs = tokenizer(email_text, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1)
    return "spam" if prediction.item() == 1 else "ham"  # 'ham' es no spam

# Probar con un correo electrónico
email_text = "Free money! Claim your reward now!"
prediction = predict_email(email_text)
print(f"El correo es: {prediction}")

El correo es: ham
